# Alocação enfermeiro-quarto por turno (NRA)

<a href="https://colab.research.google.com/github/TaygoCezar/Matematica_Computacional_TrabalhoFinal/blob/main/NRA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a> 


# Instalação dos pacotes necessários

In [ ]:
%pip install -q pulp
%pip install -q numpy
%pip install -q matplotlib
%pip install -q pandas

# Import dos pacotes necessários

In [ ]:
import pulp
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import time
import os
import random
import copy
import sys


## Definição dos caminhos para as instâncias e constantes

In [ ]:
I04: str = "resources/i04"
I06: str = "resources/i06"
SEED: int = 467

## Programação Linear Inteira (PLI)

In [ ]:
def solve_nra_problem(instance_dir):
    # start of time measurement
    start_time = time.perf_counter()

    # data load
    with open(f"{instance_dir}/instance_info.json", "r") as f:
        info: dict = json.load(f)

    nurse_shifts: pd.DataFrame = pd.read_csv(f"{instance_dir}/nurse_shifts.csv")
    room_shifts: pd.DataFrame = pd.read_csv(f"{instance_dir}/occupied_room_shifts.csv")

    # extract penalties weights
    w_skill: int = info["weights"]["S2_room_nurse_skill"]
    w_work: int = info["weights"]["S4_nurse_excessive_workload"]

    # init the solver
    prob = pulp.LpProblem("NRA_Optimization", pulp.LpMinimize)

    # binary variables: 1 if the nurse is allocated to the room, otherwise 0
    x_vars = {}  # x[n, r, s]

    # integer variables: nurse workload
    e_vars = {}  # e[n, s]

    objective_terms = []

    # get the each turn
    shifts_unique = room_shifts["global_shift"].unique()

    # for each turn: 0, 1, ..., 41
    for s in shifts_unique:
        rooms_in_shift = room_shifts[room_shifts["global_shift"] == s]
        available_nurses = nurse_shifts[nurse_shifts["global_shift"] == s]

        if available_nurses.empty and not rooms_in_shift.empty:
            continue
        
        # for each nurse
        for _, nurse in available_nurses.iterrows():
            n_id: str = nurse["nurse_id"]
            n_skill: int = nurse["skill_level"]
            n_capacity: int = nurse["max_load"]  

            e_vars[(n_id, s)] = pulp.LpVariable(
                f"e_{n_id}_{s}", lowBound=0, cat=pulp.LpInteger
            )

            # add the penalty weight
            objective_terms.append(w_work * e_vars[(n_id, s)])

            workload_sum: int = 0

            for _, room in rooms_in_shift.iterrows():
                r_id = room["room_id"]
                r_req_skill = room["max_skill_required"]
                r_workload = room["total_room_workload"]

                # binary variable
                x_vars[(n_id, r_id, s)] = pulp.LpVariable(
                    f"x_{n_id}_{r_id}_{s}", cat=pulp.LpBinary
                )

                # skill deficit
                deficit = max(0, r_req_skill - n_skill)

                # add the deficit
                if deficit > 0:
                    objective_terms.append(w_skill * deficit * x_vars[(n_id, r_id, s)])

                # total workload
                workload_sum += r_workload * x_vars[(n_id, r_id, s)]

            # soft constraint: workload
            prob += (
                workload_sum - n_capacity <= e_vars[(n_id, s)],
                f"Workload_{n_id}_{s}",
            )

        # hard constraint: room coverage 
        for _, room in rooms_in_shift.iterrows():
            r_id = room["room_id"]
            prob += (
                pulp.lpSum(
                    [
                        x_vars[(n["nurse_id"], r_id, s)]
                        for _, n in available_nurses.iterrows()
                    ]
                )
                == 1, # c_1 + c_2 + ... + c_n = 1
                f"Coverage_{r_id}_{s}",
            )

    # calculate all penalties
    prob += pulp.lpSum(objective_terms), "Total_Penalties"

    # solve with CBC
    prob.solve(pulp.PULP_CBC_CMD(msg=True))

    # end time measurement
    end_time: float = time.perf_counter()
    execution_time: float = end_time - start_time

    # results
    total_penalty = pulp.value(prob.objective)
    
    print("\n" + "="*50)
    print("RESULTADOS DO MÉTODO EXATO (PLI - Solver CBC)")
    print("="*50)
    print(f"Status da Resolução                 : {pulp.LpStatus[prob.status]}")
    print(f"Função Objetivo (Penalidade Total)  : {total_penalty}")
    print(f"Tempo de Processamento              : {execution_time:.2f} segundos")
    print("="*50)

    # save in a json
    results = 'resultados_comparativos.json'
    
    if os.path.exists(results):
        with open(results, 'r', encoding='utf-8') as f:
            json_data = json.load(f)
    else:
        json_data = {
            "PLI": {},
            "GA": {
                "penalidade_melhor": 0.0,
                "penalidade_media": 0.0,
                "tempo_medio_segundos": 0.0
            }
        }
    
    # update the PLI keys with the exact results
    json_data["PLI"]["penalidade_total"] = float(total_penalty) if total_penalty is not None else 0.0
    json_data["PLI"]["tempo_segundos"] = float(execution_time)

    # write back to the file
    with open(results, 'w', encoding='utf-8') as f:
        json.dump(json_data, f, indent=4)

    print(f"-> Arquivo '{results}' atualizado com as métricas do PLI\n")

    return prob

### Execução do PLI na instância i04

In [ ]:
solve_nra_problem(I04)

### Execução do PLI na instância i06

In [ ]:
solve_nra_problem(I06)

## Metaheurística: Algoritmo Genético

In [ ]:
class NRA_Environment:
    """
    Class to load and encapsulate the problem data for a specific shift.
    """

    def __init__(
        self, rooms_df: pd.DataFrame, nurses_df: pd.DataFrame, w_skill: int, w_work: int
    ):
        self.rooms = rooms_df.to_dict("records")
        self.nurses: dict[str, dict] = {
            n["nurse_id"]: n for n in nurses_df.to_dict("records")
        }
        self.nurse_ids = list(self.nurses.keys())
        self.w_skill: int = w_skill
        self.w_work: int = w_work

    def calculate_fitness(self, chromosome: list[str]) -> int:
        """
        Calculate the total penalty. Less is better.
        """
        penalty: int = 0
        workload_sum: dict[str, int] = {n_id: 0 for n_id in self.nurse_ids}

        # room (gene index) and nurse (gene value)
        for room_idx, nurse_id in enumerate(chromosome):
            room: dict = self.rooms[room_idx]
            nurse: dict = self.nurses[nurse_id]

            # hability deficit penalty
            deficit = max(0, room["max_skill_required"] - nurse["skill_level"])
            penalty += self.w_skill * deficit

            # Acumular carga de trabalho para este enfermeiro
            workload_sum[nurse_id] += room["total_room_workload"]

        # workload penalty
        for nurse_id, total_load in workload_sum.items():
            nurse: dict = self.nurses[nurse_id]
            excess: int = max(0, total_load - nurse["max_load"])
            penalty += self.w_work * excess

        return penalty


class GeneticAlgorithm:
    def __init__(
        self,
        env: NRA_Environment,
        seed: int = 42,
        pop_size: int = 100,
        mutation_rate: float = 0.1,
        generations: int = 200,
        tournament_size: int = 3,
    ):
        self.env: NRA_Environment = env
        self.pop_size: int = pop_size
        self.mutation_rate: float = mutation_rate
        self.generations: int = generations
        self.tournament_size: int = tournament_size
        self.num_genes: int = len(self.env.rooms)
        self.history_best: list[int | float] = (
            []
        )
        random.seed(seed)

    def create_individual(self) -> list[str]:
        """
        Create a valid cromossome
        """
        # Chromosomes are lists of strings that represent which nurse is in the room
        # the list index represents the room
        return [random.choice(self.env.nurse_ids) for _ in range(self.num_genes)]

    def crossover(self, parent1: list[str], parent2: list[str]) -> list[str]:
        """
        Randomly choose who will inherit the gene (nurse in a room).
        """
        child: list[str] = []
        for i in range(self.num_genes):
            if random.random() < 0.5:
                child.append(parent1[i])
            else:
                child.append(parent2[i])
        return child

    def mutate(self, individual: list[str]) -> list[str]:
        """
        Randomly choose a room and change the nurse to other avaiable nurse.
        """
        for i in range(self.num_genes):
            if random.random() < self.mutation_rate:
                individual[i] = random.choice(self.env.nurse_ids)
        return individual

    def tournament_selection(self, population, fitnesses) -> list[str]:
        """
        Select the best cromossome between 'k' randomly chosen.
        """
        # list with "k" penalties of random cromossomes
        selected_indices: list[int] = random.sample(
            range(self.pop_size), self.tournament_size
        )
        best_idx: int = min(selected_indices, key=lambda idx: fitnesses[idx])

        # returns the best cromossome
        return population[best_idx]

    def run(self):
        # init the population
        population: list[list[str]] = [
            self.create_individual() for _ in range(self.pop_size)
        ]

        best_overall_fitness = float("inf")
        best_overall_individual = None

        for gen in range(self.generations):
            fitnesses: list[int] = [
                self.env.calculate_fitness(ind) for ind in population
            ]

            # store the best iteration
            current_best_fit: int = min(fitnesses)
            if current_best_fit < best_overall_fitness:
                best_overall_fitness = current_best_fit
                best_overall_individual = copy.deepcopy(
                    population[fitnesses.index(current_best_fit)]
                )

            # add the best population
            self.history_best.append(best_overall_fitness)

            # keep the absolute best indiviual
            new_population: list[list[str]] = [best_overall_individual]  # type: ignore

            # make a new population
            while len(new_population) < self.pop_size:
                # get the best element with tournament to do crossover and mutation
                parent_1: list[str] = self.tournament_selection(population, fitnesses)
                parent_2: list[str] = self.tournament_selection(population, fitnesses)

                # cross the bests
                child: list[str] = self.crossover(parent_1, parent_2)

                # mutation
                child = self.mutate(child)

                # add the new cromossome in the population
                new_population.append(child)

            # update the population
            population = new_population

        return best_overall_individual, best_overall_fitness


def solve_nra_metaheuristic(
        instance_dir: str, 
        seed: int = 42, 
        repetitions: int = 5, 
        pop_size: int = 50, 
        mutation_rate: float = 0.15, 
        generations: int = 100,
        tournament_size: int = 3
        ):
    # start of time measurement
    start_time = time.perf_counter()

    # random seed
    current_seed: int = seed

    # data load
    with open(f"{instance_dir}/instance_info.json", "r") as f:
        info = json.load(f)

    nurse_shifts: pd.DataFrame = pd.read_csv(f"{instance_dir}/nurse_shifts.csv")
    room_shifts: pd.DataFrame = pd.read_csv(f"{instance_dir}/occupied_room_shifts.csv")

    # weights
    w_skill: int = info["weights"]["S2_room_nurse_skill"]
    w_work: int = info["weights"]["S4_nurse_excessive_workload"]

    shifts_unique = room_shifts["global_shift"].unique()

    penalties_per_rep =[0] * repetitions
    time_per_rep: list[float] = [0.0] * repetitions
    representative_convergence = []
    total_penalty_all_shifts = 0

    print(
        f"Executando Metaheurística ({repetitions} repetições independentes por turno)..."
    )

    # for each unique shift (s)
    for s in shifts_unique:
        rooms_in_shift = room_shifts[room_shifts["global_shift"] == s]
        available_nurses = nurse_shifts[nurse_shifts["global_shift"] == s]

        if available_nurses.empty or rooms_in_shift.empty:
            continue

        env: NRA_Environment = NRA_Environment(rooms_in_shift, available_nurses, w_skill, w_work)  # type: ignore

        best_shift_fitness: float | int = float("inf")
        convergence_history: list = []

        # run "r" repetitions to analise
        for r in range(repetitions):
            # start rep time measurement
            start_rep: float = time.perf_counter()

            ga = GeneticAlgorithm(
                env, 
                seed=current_seed, 
                pop_size=pop_size, 
                mutation_rate=mutation_rate, 
                generations=generations,
                tournament_size=tournament_size
            )
            _, fitness = ga.run()

            end_rep: float = time.perf_counter()
            time_per_rep[r] += (end_rep - start_rep)

            penalties_per_rep[r] += fitness

            if fitness < best_shift_fitness:
                best_shift_fitness = fitness
                # keep the best learn curve
                convergence_history = (
                    ga.history_best
                )

            current_seed += 1

        if len(representative_convergence) == 0 and convergence_history:
            representative_convergence = convergence_history
        total_penalty_all_shifts += best_shift_fitness

    end_time_global = time.perf_counter()
    execution_time_total = end_time_global - start_time
    print(f"Melhor Penalidade Total Encontrada (GA): {total_penalty_all_shifts}")

    # Graphics
    attempts = range(1, repetitions + 1)
    avarage_penalty = np.mean(penalties_per_rep)
    best_penalty = np.min(penalties_per_rep)
    avarage_time = np.mean(time_per_rep)
    std_penalties = np.std(penalties_per_rep)

    # variability graphic
    fig1, ax1 = plt.subplots(figsize=(10, 6))
    ax1.plot(attempts, penalties_per_rep, marker='o', linestyle='-', color='#9ecae1', 
             markerfacecolor='#3182bd', markersize=8, linewidth=1.5, zorder=3, 
             label='Penalidade da Execução')
    
    ax1.axhline(y=avarage_penalty, color='orange', linestyle='--', linewidth=2, zorder=2, 
                label=f'Média das Tentativas: {avarage_penalty:.2f}')
    
    ax1.axhline(y=best_penalty, color='green', linestyle='--', linewidth=2, zorder=1, 
                label=f'Recorde (Melhor Escala): {best_penalty}')

    ax1.errorbar(attempts, penalties_per_rep, yerr=std_penalties, fmt='none', 
             ecolor='purple', capsize=5, elinewidth=1.5, zorder=2, 
             label=f'Desvio Padrão: {std_penalties:.2f}')
    
    ax1.set_title('Impacto da Aleatoriedade no Algoritmo Genético\n(Comparação do Resultado Final entre as Diferentes Execuções)', 
              fontsize=14, fontweight='bold', pad=15)
    ax1.set_xlabel('Número da Execução (Tentativa Independente)', fontsize=12)
    ax1.set_ylabel('Penalidade Total do Período', fontsize=12)
    ax1.set_xticks(attempts)
    # ax1.grid(True, linestyle=':', alpha=0.7, color='gray')
    ax1.legend(loc='upper right', framealpha=0.9)
    plt.tight_layout()
    plt.savefig(f'{instance_dir}/variabilidade_ga.png', dpi=300)
    plt.show()
    plt.close(fig1)

    # convergence graphic
    if representative_convergence:
        plt.figure(figsize=(10, 5))
        plt.plot(representative_convergence, color='red', linewidth=2, label='Melhor Indivíduo (Fitness)')
        plt.title('Histórico de Convergência (Amostra de um Turno)', fontsize=14)
        plt.xlabel('Gerações', fontsize=12)
        plt.ylabel('Penalidade', fontsize=12)
        plt.grid(True, linestyle='--', alpha=0.7)
        plt.legend()
        plt.savefig(f'{instance_dir}/convergencia_ga.png', dpi=300, bbox_inches='tight')
        plt.show()
        plt.close()
    
    print("\n-> Gráficos 'boxplot_ga.png' e 'convergencia_ga.png' salvos com sucesso!")

    # save results
    Results = 'resultados_comparativos.json'
    
    # check if the file already exists
    if os.path.exists(Results):
        with open(Results, 'r', encoding='utf-8') as f:
            json_data = json.load(f)
    else:
        json_data = {
            "PLI": {
                "penalidade_total": 0.0,
                "tempo_segundos": 0.0
            },
            "GA": {}
        }
    
    # update the GA keys with the fresh results from this run
    json_data["GA"]["penalidade_melhor"] = float(best_penalty)
    json_data["GA"]["penalidade_media"] = float(avarage_penalty)
    json_data["GA"]["tempo_medio_segundos"] = float(avarage_time)

    # write back in the file
    with open(Results, 'w', encoding='utf-8') as f:
        json.dump(json_data, f, indent=4)
    
    # print data table
    print("\n" + "="*50)
    print("ANÁLISE DE RESULTADOS E ESTATÍSTICAS OFICIAIS (GA)")
    print("="*50)
    print(f"Melhor Escala Completa Encontrada         : {best_penalty}")
    print(f"Pior Escala Completa Encontrada           : {np.max(penalties_per_rep)}")
    print(f"Média das Execuções (Comportamento Médio) : {avarage_time:.2f}")
    print(f"Desvio Padrão (Variabilidade/Robustez)    : {np.std(penalties_per_rep):.2f}")
    print("-" * 50)
    print(f"Tempo Médio por Execução Completa         : {avarage_time:.2f} segundos")
    print(f"Tempo Total Gasto no Teste (15 rodadas)   : {execution_time_total:.2f} segundos")
    print("="*50)
    print("Gráficos do GA gerados com sucesso")
    print(f"-> Arquivo '{Results}' atualizado com as métricas do GA")

    return best_penalty


### Usando o algoritmo genético na instância i04

In [ ]:
solve_nra_metaheuristic(I04, SEED, 15)

### Usando o algoritmo genético na instância i06

In [ ]:
solve_nra_metaheuristic(I06, SEED, 15)

## Gráfico comparativo entre PLI e AG

In [ ]:
RESULT_FILE = 'resultados_comparativos.json'

def load_data() -> dict:
    if not os.path.exists(RESULT_FILE):
        print(f"\n[ERRO] O arquivo '{RESULT_FILE}' não foi encontrado.")
        sys.exit()

    with open(RESULT_FILE, 'r', encoding='utf-8') as f:
        return json.load(f)

def generate_comparative_graphic():
    # Loads data in a completely dynamic way (No hardcoded numbers in the code)
    data = load_data()
    
    # Extract the data from the PLI (using .get to avoid errors if something is missing)
    penality_pli: float = data["PLI"].get("penalidade_total", 0.0)
    time_pli_seconds: float = data["PLI"].get("tempo_segundos", 0.0)
    
    # Extract the data from GA.
    penality_ga_best: float = data["GA"].get("penalidade_melhor", 0.0)
    penality_ga_avarage: float = data["GA"].get("penalidade_media", 0.0)
    time_ga_avarage: float = data["GA"].get("tempo_medio_segundos", 0.0)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6))

    labels_quality: list[str] = ['PLI\n(Ótimo Matemático)', 'GA\n(Melhor Caso)', 'GA\n(Média das Execuções)']
    values_quality: list[float] = [penality_pli, penality_ga_best, penality_ga_avarage]
    colors_quality: list[str] = ['#6e9995', '#6f6e99', "#1D402F"]

    bars1 = ax1.bar(labels_quality, values_quality, color=colors_quality, edgecolor='black', zorder=3)
    ax1.set_title('Comparação de Qualidade (Função Objetivo)', fontsize=12, fontweight='bold', pad=15)
    ax1.set_ylabel('Penalidade Total', fontsize=11)
    ax1.grid(axis='y', linestyle=':', alpha=0.7, zorder=0)

    for bar in bars1:
        yval = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width()/2, yval + (max(values_quality)*0.02), 
                 round(yval, 2), ha='center', va='bottom', fontweight='bold')

    labels_time: list[str] = ['PLI\n(Solver Exato)', 'GA\n(Metaheurística)']
    values_time: list[float] = [time_pli_seconds, time_ga_avarage]
    colors_time: list[str] = ['#b26810', '#bfb372']

    bars2 = ax2.bar(labels_time, values_time, color=colors_time, edgecolor='black', width=0.5, zorder=3)
    ax2.set_title('Tempo de Execução', fontsize=12, fontweight='bold', pad=15)
    ax2.set_ylabel('Tempo em Segundos', fontsize=11)
    ax2.grid(axis='y', linestyle=':', alpha=0.7, zorder=0)

    for bar in bars2:
        yval = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width()/2, yval + (max(values_time)*0.02), 
                 f"{round(yval, 2)}s", ha='center', va='bottom', fontweight='bold')

    plt.suptitle('Análise Comparativa Geral: Programação Linear Inteira vs. Algoritmo Genético', 
                 fontsize=16, fontweight='bold', y=1.05)
    plt.tight_layout()
    plt.savefig('comparativo_pli_vs_ga.png', dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()

    print("\n" + "="*50)
    print("GRÁFICO COMPARATIVO GERADO COM SUCESSO!")
    print("="*50)
    print(f"-> Arquivo lido  : {RESULT_FILE}")
    print(f"-> Imagem salva  : comparativo_pli_vs_ga.png")
    print("="*50 + "\n")

generate_comparative_graphic()

# Teste de sensibilidade do GA aos parâmetros

### Teste de sensibilidade à variação da taxa de mutação:
Taxa de mutação de 5%:

In [ ]:
solve_nra_metaheuristic(I04, SEED, 15, pop_size=50, mutation_rate=0.05, generations=100, tournament_size=3)

Usando uma taxa de mutação de 10%:

In [ ]:
solve_nra_metaheuristic(I04, SEED, 15, pop_size=50, mutation_rate=0.10, generations=100, tournament_size=3)

Usando uma taxa de mutação de 15%:

In [ ]:
solve_nra_metaheuristic(I04, SEED, 15, pop_size=50, mutation_rate=0.15, generations=100, tournament_size=3)

# Teste de sensibilidade à variação da população
Usando uma população de tamanho 20:

In [ ]:
solve_nra_metaheuristic(I04, SEED, 15, pop_size=20, mutation_rate=0.15, generations=100, tournament_size=3)

Usando uma população de tamanho 50:

In [ ]:
solve_nra_metaheuristic(I04, SEED, 15, pop_size=50, mutation_rate=0.15, generations=100, tournament_size=3)

Usando uma populção de tamanho 100:

In [ ]:
solve_nra_metaheuristic(I04, SEED, 15, pop_size=100, mutation_rate=0.15, generations=100, tournament_size=3)